<a href="https://colab.research.google.com/github/idoco34/Alzheimer1/blob/main/Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install anvil-uplink
!pip install --upgrade anvil-uplink

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.8/133.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.2 MB/s eta 0:00:00


In [ ]:
#import section
import anvil.server
import anvil._threaded_server
import anvil.server
import tensorflow as tf
import numpy as np
import io
import anvil.server
from PIL import Image
from google.colab import drive

UPLINK_KEY = "server_LWC5SO2RUHGLGXDQCBZPM4KW-ONFNFYYEZLBJ6766"

In [ ]:
# Workaround for problem with anvil
class DummyTracer:
    def start_as_current_span(self, name):
        from contextlib import contextmanager
        @contextmanager
        def dummy(): yield
        return dummy()

# Replace the tracer function with our dummy
anvil._threaded_server.ensure_anvil_tracer = lambda: DummyTracer()

In [ ]:
# Load model
drive.mount('/drive')
model = tf.keras.models.load_model('/drive/MyDrive/models/model.keras')

ValueError: mount failed

In [ ]:
#echo server for debugging purposes
@anvil.server.callable
def server_says_hello(input_text):
    return f"Echo: {input_text}"

print("Connecting to Anvil...")
anvil.server.connect(UPLINK_KEY)
print("Echo Server is online and authenticated.")

# This keeps the script running so it can listen for calls
#anvil.server.wait_forever()

In [ ]:
IMG_HEIGHT = 128
IMG_WIDTH = 128

def preprocess_image(feature_dict):
    """Decodes, resizes, and normalizes an image from a feature dictionary."""
    # Get the image bytes from the dictionary
    image_bytes = feature_dict['bytes']

    # Decode the image
    image = tf.image.decode_jpeg(image_bytes, channels=3) # Decode as color image (3 channels)

    # Resize the image
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])

    # Normalize the image to the range [0, 1]
    image = image / 255.0

    return image

In [ ]:
@anvil.server.callable
def predict_image(img):

    img_array = np.array(
      Image.open(io.BytesIO(img.get_bytes()))
      .convert('RGB')
      .resize((128, 128))
    )

    # Normalize the image to the range [0, 1]
    img_array = img_array.astype('float32') / 255.0

    single_image_input = np.expand_dims(img_array, axis=0)  # Add batch dimension

    predictions = model.predict(single_image_input, verbose=0) # Added verbose=0 to suppress output
    class_names = ['Mild_Demented', 'Moderate_Demented', 'Non_Demented', 'Very_Mild_Demented']

    # Get the index of the highest value
    ind = np.argmax(predictions[0])
    predicted_class_name = class_names[ind]

    # Return the predicted class name and its probability
    return f"This image most likely belongs to the {predicted_class_name} class. Probability: {predictions[0][ind]:.4f}"


In [ ]:
@anvil.server.callable
def predict_image(img):

    # Step 1-6: Get bytes, open image, convert to RGB, resize, and reshape
  img_array = np.array(
      Image.open(io.BytesIO(img.get_bytes()))
      .convert('RGB')
      .resize((128, 128))
  ).reshape((128, 128, 3))  # Reshape to (128, 128, 3)

  single_image_input = np.expand_dims(img_array, axis=0)

  predictions = model.predict(single_image_input)
  mild = predictions[0][0]
  moderate = predictions[0][1]
  non_demented = predictions[0][2]
  very_mild = predictions[0][3]

  # Get the index of the highest value
  ind = np.argmax(predictions[0])

  if ind == 0:
    return "This image most likely belongs to the Mild Demented class. " + str(mild)
  if ind == 1:
    return "This image most likely belongs to the Moderate Demented class." + str(moderate)
  if ind == 2:
    return "This image most likely belongs to the Non Demented class." + str(non_demented)
  return "This image most likely belongs to the Very Mild Demented class." + str(very_mild)